In [1]:
## Uncomment and run the below to commands for reloading the modules of un_llm_analysis after changes. 
# %load_ext autoreload
# %autoreload 2

# Issue Analysis on the UN Daily Briefings

This notebook visualizes how the proportion of a set of selected issues changes over time.

If an LLM determines that a selected issue is relevant to a section, the corresponding issue-section mapping is saved to the database.

## 1. Set up and load data

This notebook uses the pre-uploaded database specified by `BRIEFINGS_DB_PATH`. 

Alternatively, the database can be created from scratch for a desired date range:
```bash
briefings update 2026-09-01 2026-09-10
briefings init
briefings load 2026-09-01 2026-09-10
```

In [2]:
from datetime import datetime, timezone

import pandas as pd
from tqdm import tqdm

from un_llm_analysis.config import BRIEFINGS_DB_PATH
from un_llm_analysis.database import (
    get_db_connection, 
    get_briefing_id_from_date, 
    get_briefings_between_dates,
    create_database,
    insert_into_issue,
    get_issues_between_dates,
)

In [3]:
# Create database and make sure the tables are created.
create_database()

2026-10-04 23:32:56.145 | INFO     | un_llm_analysis.database:create_database:48 - Database created in "/home/yhs/projects/correlaid/correlaid-mentoring-project/app/briefings.db"


In [4]:
# Get the data in the database within the specified date range
start_date = "2026-09-01"
end_date = "2026-09-05"

with get_db_connection(BRIEFINGS_DB_PATH) as conn:
    briefings = get_briefings_between_dates(conn, start_date, end_date)

In [5]:
# Get the section data from the briefings and store it in a DataFrame

rows = []

for briefing in tqdm(briefings):
    briefing_id = get_briefing_id_from_date(briefing.briefing_date)
    for i, section in enumerate(briefing.sections, start=1):
        section_id = briefing_id * 100 + i
        rows.append(
            {
                "section_id": section_id,
                "briefing_id": briefing_id,
                "section_title": section.title,
                "section_text": section.text,
            }
        )

section_data = pd.DataFrame(rows)

section_data.info()
section_data.head(2)

100%|██████████| 5/5 [00:00<00:00, 4719.06it/s]

<class 'pandas.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   section_id     49 non-null     int64
 1   briefing_id    49 non-null     int64
 2   section_title  49 non-null     str  
 3   section_text   49 non-null     str  
dtypes: int64(2), str(2)
memory usage: 58.1 KB


,section_id,briefing_id,section_title,section_text
0,2026090101,20260901,TODAY’S BRIFING,"At 12:30 p.m., Jérôme Bonnafont, the Permanent..."
1,2026090102,20260901,TOMORROW’S GUEST,"Inger Andersen, the Executive Director of UNEP..."


## 2. Issue Mapping

In [6]:
selected_issues = {
    "ISSUE-1": "Israel-Palestine conflict, Gaza and the Occupied Palestinian Territory",
    "ISSUE-2": "War in Ukraine and its direct security and humanitarian consequences",
    "ISSUE-3": "Sudan conflict, political process and humanitarian crisis",
    "ISSUE-4": "Democratic Republic of the Congo conflict and humanitarian crisis",
    "ISSUE-5": "Yemen conflict and humanitarian crisis",
    "ISSUE-6": "South Sudan conflict, peace process and humanitarian crisis",
    "ISSUE-7": "Syria conflict and humanitarian crisis",
    "ISSUE-8": "Haiti political, security and humanitarian crisis",
    "ISSUE-9": "Afghanistan political and humanitarian crisis",
    "ISSUE-10": "Myanmar conflict and Rohingya crisis",
    "ISSUE-11": "Central African Republic conflict and political transition",
    "ISSUE-12": "Lebanon political and humanitarian crisis and Israel-Lebanon hostilities",
    "ISSUE-13": "Ethiopia conflict and humanitarian crisis",
    "ISSUE-14": "Libya political transition and insecurity",
    "ISSUE-15": "Sahel and Lake Chad Basin conflict and insecurity",
    "ISSUE-16": "Somalia conflict and humanitarian crisis",
    "ISSUE-17": "Climate change and long-term environmental issues",
    "ISSUE-18": "Natural disasters and acute emergency response",
    "ISSUE-19": "Global health, disease outbreaks and vaccination",
    "ISSUE-20": "Food security, hunger and malnutrition as a primary or cross-country issue",
    "ISSUE-21": "Refugees, displacement and migration as a primary or cross-border issue",
    "ISSUE-22": "Human rights, gender equality and child protection as a primary or cross-country issue",
    "ISSUE-23": "Sustainable development and development financing",
    "ISSUE-24": "Disarmament, arms control and non-proliferation",
}

issue_text = ""
for issue in selected_issues:
    issue_text += f"* {issue} {selected_issues[issue]}\n"

issue_definitions = issue_text.strip()
print(issue_definitions)

* ISSUE-1 Israel-Palestine conflict, Gaza and the Occupied Palestinian Territory
* ISSUE-2 War in Ukraine and its direct security and humanitarian consequences
* ISSUE-3 Sudan conflict, political process and humanitarian crisis
* ISSUE-4 Democratic Republic of the Congo conflict and humanitarian crisis
* ISSUE-5 Yemen conflict and humanitarian crisis
* ISSUE-6 South Sudan conflict, peace process and humanitarian crisis
* ISSUE-7 Syria conflict and humanitarian crisis
* ISSUE-8 Haiti political, security and humanitarian crisis
* ISSUE-9 Afghanistan political and humanitarian crisis
* ISSUE-10 Myanmar conflict and Rohingya crisis
* ISSUE-11 Central African Republic conflict and political transition
* ISSUE-12 Lebanon political and humanitarian crisis and Israel-Lebanon hostilities
* ISSUE-13 Ethiopia conflict and humanitarian crisis
* ISSUE-14 Libya political transition and insecurity
* ISSUE-15 Sahel and Lake Chad Basin conflict and insecurity
* ISSUE-16 Somalia conflict and humanitaria

In [7]:
prompt_version = "v4"
prompt = """**Task**

Identify the most relevant issue for the text below.

**Issue Definitions**

{issue_definitions}

**Text**

{section_text}

**Decision Rules**

- Assign one issue by default.
- Return None if none of the issue definitions clearly apply.
- Assign two issues only if the text contains two distinct substantive topics and each independently matches an issue definition.
- Never assign more than two issues.
- Do not assign a second issue only because it is mentioned as background, consequence, or context.

**Output Format**

Return exactly one of the following forms:

None
['ISSUE-<number>']
['ISSUE-<number>', 'ISSUE-<number>']

Do not return ['None'].
Do not include any explanation or additional text.
"""


### Ollama and mlflow

Set up Ollama and mlflow to run local LLMs and capture detailed tracing records.

```bash
# Download LLMs to test
ollama pull llama3.2
ollama pull gemma3:4b
# Other models:  qwen3.5:4b llama3.2:1b qwen3:1.7b

# Install dependencies to use mlflow with local Ollama
pipenv shell
pipenv install mlflow openai 
# Or sync dependencies from Pipfile: `pipenv install --dev`

# Start a local mlflow server
mlflow server
# Run it in the background: `nohup mlflow server --port 5000 > mlflow.log 2>&1 &`
```

In [8]:
import mlflow
from openai import OpenAI

mlflow.openai.autolog()
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Issue Mapping")

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="dummy",
)

In [ ]:
# This cell is the main loop for running the issue mapping experiments with the LLM.
# More models can be added for comparison if needed.

# Here we use `qwen3.5:4b` because it produced the best results for this task among the models tested below.
# models = ["llama3.2:latest", "gemma3:4b", "qwen3.5:4b", "llama3.2:1b", "qwen3:1.7b"]
models = ["qwen3.5:4b"]

conn = get_db_connection()

# Common generation settings
temperature = 0
seed = 42
reasoning_effort = "none"
max_completion_tokens = 128

for llm in models:
    with mlflow.start_run(run_name=llm):
        # Log model-level experiment settings
        mlflow.log_params(
            {
                "model": llm,
                "prompt_version": prompt_version,
                "temperature": temperature,
                "seed": seed,
                "reasoning_effort": reasoning_effort,
                "max_completion_tokens": max_completion_tokens,
                "n_sections": len(section_data),
            }
        )

        for _, section in section_data.iterrows():

            section_text = (
                f"**{section.section_title}**\n\n"
                f"{section.section_text}"
            )

            response = client.chat.completions.create(
                model=llm,
                messages=[{
                    "role": "user",
                    "content": prompt.format(
                        section_text=section_text,
                        issue_definitions=issue_definitions,
                    ),
                }],
                temperature=temperature,
                seed=seed,
                reasoning_effort=reasoning_effort,
                max_completion_tokens=max_completion_tokens,
            )

            # Set the section_id as a trace tag
            trace_id = mlflow.get_last_active_trace_id(thread_local=True)
            mlflow.flush_trace_async_logging()
            if trace_id is not None:
                mlflow.set_trace_tag(
                    trace_id,
                    "section_id",
                    str(section.section_id),
                )

            insert_into_issue(
                conn=conn,
                briefing_id=section.briefing_id,
                section_id=section.section_id,
                issues=response.choices[0].message.content,
                model=llm,
                created_at=datetime.fromtimestamp(response.created, tz=timezone.utc).isoformat(timespec="seconds"),
            )

conn.close()

## 3. Visualization

### 1) Build the DataFrame for analysis

In [9]:
# Get the issue data from the database within the specified date range

with get_db_connection() as conn:
    issue_data = get_issues_between_dates(conn=conn, start_date=start_date, end_date=end_date)

issue_data["date"] = pd.to_datetime(issue_data["briefing_id"].astype(str), format="%Y%m%d")

issue_data.head(2)

,briefing_id,section_id,issue,model,created_at,date
0,20260902,2026090201,ISSUE-17,qwen3.5:4b,2026-09-23T14:53:05+00:00,2026-09-02
1,20260902,2026090202,ISSUE-12,qwen3.5:4b,2026-09-23T14:54:02+00:00,2026-09-02


In [10]:
# Store the length of each section
section_df = section_data.copy()
section_df["section_length"] =  section_df["section_text"].str.len()

# Calculate total length of the sections on each date
total_length = (
    section_df.groupby("briefing_id")["section_length"]
    .sum()
    .reset_index(name="total_length")
)

section_df = section_df.merge(total_length, on="briefing_id", how="left")

section_df.head(2)

,section_id,briefing_id,section_title,section_text,section_length,total_length
0,2026090101,20260901,TODAY’S BRIFING,"At 12:30 p.m., Jérôme Bonnafont, the Permanent...",402,14993
1,2026090102,20260901,TOMORROW’S GUEST,"Inger Andersen, the Executive Director of UNEP...",400,14993


In [11]:
# Merge the issue data with the section data to get the section lengths and total lengths for each issue
issue_sections = issue_data.merge(
    section_df[["section_id", "section_length", "total_length"]],
    on="section_id",
    how="left"
)

issue_sections.head(5)

,briefing_id,section_id,issue,model,created_at,date,section_length,total_length
0,20260902,2026090201,ISSUE-17,qwen3.5:4b,2026-09-23T14:53:05+00:00,2026-09-02,854,13262
1,20260902,2026090202,ISSUE-12,qwen3.5:4b,2026-09-23T14:54:02+00:00,2026-09-02,1737,13262
2,20260902,2026090202,ISSUE-20,qwen3.5:4b,2026-09-23T14:54:02+00:00,2026-09-02,1737,13262
3,20260902,2026090203,ISSUE-1,qwen3.5:4b,2026-09-23T14:55:23+00:00,2026-09-02,2649,13262
4,20260902,2026090203,ISSUE-20,qwen3.5:4b,2026-09-23T14:55:23+00:00,2026-09-02,2649,13262


In [12]:
# Calculate the total length of sections for each issue on each date
issue_length = (
    issue_sections
    .groupby(["date", "issue"])["section_length"]
    .sum()
    .reset_index(name="issue_length")
)

issue_length.head(2)

,date,issue,issue_length
0,2026-09-01,ISSUE-1,3965
1,2026-09-01,ISSUE-12,969


In [13]:
# Calculate the frequency of sections for each issue on each date
frequency = (
    issue_sections
    .groupby(["date", "issue"])["section_id"]
    .nunique()
    .reset_index(name="frequency")
)

frequency.head(2)

,date,issue,frequency
0,2026-09-01,ISSUE-1,2
1,2026-09-01,ISSUE-12,1


In [14]:
# Build a complete date x issue grid.
issues = [f"ISSUE-{i}" for i in range(1, 25)]

dates = pd.date_range(start_date, end_date, freq="D")

issue_date_grid = (
    pd.MultiIndex.from_product(
        [dates, issues],
        names=["date", "issue"]
    )
    .to_frame(index=False)
)

issue_date_grid.head(2)

,date,issue
0,2026-09-01,ISSUE-1
1,2026-09-01,ISSUE-2


In [15]:
# Merge the issue date grid with the issue length and frequency data to get a complete dataset for analysis

issue_stats = issue_date_grid.merge(
    issue_sections[["date", "total_length"]].drop_duplicates(),
    on="date",
    how="left"
)

issue_stats = issue_stats.merge(
    issue_length,
    on=["date", "issue"],
    how="left" 
)

issue_stats = issue_stats.merge(
    frequency,
    on=["date", "issue"],
    how="left"
)

issue_stats = (
    issue_stats
    .fillna({"issue_length": 0, "frequency": 0, "total_length": 0})
    .astype({"issue_length": int, "frequency": int, "total_length": int})
)

issue_stats.head()

,date,issue,total_length,issue_length,frequency
0,2026-09-01,ISSUE-1,14993,3965,2
1,2026-09-01,ISSUE-2,14993,1638,1
2,2026-09-01,ISSUE-3,14993,1785,1
3,2026-09-01,ISSUE-4,14993,1700,1
4,2026-09-01,ISSUE-5,14993,0,0


### 2) Draw the plots

In [16]:
import plotly.express as px

fig_1 = px.line(
    issue_stats,
    x="date",
    y="frequency",
    color="issue",
    markers=False,
    title=f"Frequency of Issues in UN Briefings between {start_date} and {end_date}"
)

fig_1.show()

In [17]:
fig_2 = px.line(
    issue_stats,
    x="date",
    y="issue_length",
    color="issue",
    markers=False,
    title=f"Total Length of Sections for each Issue in UN Briefings between {start_date} and {end_date}"
)

fig_2.show()

In [18]:
issue_stats["proportion"] = (
    issue_stats["issue_length"] / issue_stats["total_length"]
).fillna(0)

fig_3 = px.line(
    issue_stats,
    x="date",
    y="proportion",
    color="issue",
    markers=False,
    title=f"Proportion of Text for each Issue in UN Briefings between {start_date} and {end_date}"
)

fig_3.show()

In [19]:
issue_counts = (
    issue_stats
    .groupby("issue")["frequency"]
    .sum()
    .reset_index(name="total_counts")
    .sort_values(by="total_counts", ascending=False)
)

fig = px.bar(
    issue_counts,
    x="total_counts",
    y="issue",
    orientation="h",
    title=f"Total Counts of Issues in UN Briefings between {start_date} and {end_date}"
)
fig.show()